# Stored execution evidence — Dog Breed Identification

This public evidence copy preserves every textual output cell supplied in `agent_final_submission.zip`.
The original notebook SHA-256 is `6d3a828994b8dea341006cb3d04680538b57c04b9bb055116892edc3a6ac08ae`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Integration Update on Colab

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `main` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on Colab.

The active flow is:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the HuggingFace dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated project runs the built-in smoke checks.


## Before running

1. In Colab, open **Secrets** and add `OPENAI_API_KEY`.
2. Add `KAGGLE_API_TOKEN` in Colab Secrets and accept the Plant Pathology competition rules on Kaggle before downloading.
3. Select a GPU runtime for real training.
4. Run the cells in order.


In [ ]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path


def normalize_repo_url(value: str) -> str:
    value = (value or "").strip()

    markdown_match = re.fullmatch(
        r"\[(.*?)\]\((https?://[^)]+)\)",
        value,
    )
    if markdown_match:
        return markdown_match.group(2)

    plain_url_match = re.search(r"https?://\S+", value)
    if plain_url_match:
        return plain_url_match.group(0)

    return value


REPO_URL = normalize_repo_url(
    os.getenv(
        "JIAOZI_REPO_URL",
        "https://github.com/Isso-W/Jiaozi.git",
    )
)
REPO_REF = os.getenv("JIAOZI_REPO_REF", "main")
REPO_DIR = Path("/content/Jiaozi")

# Important:
# Colab may still be inside /content/Jiaozi from a previous run.
# Move out before deleting and cloning the repository again.
os.chdir("/content")

if REPO_DIR.exists():
    print("Removing existing repository:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)

clone_cmd = [
    "git",
    "clone",
    "--depth",
    "1",
    "--branch",
    REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

completed = subprocess.run(
    clone_cmd,
    cwd="/content",
    capture_output=True,
    text=True,
)

print("=== git stdout ===")
print(completed.stdout)

print("=== git stderr ===")
print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        "Git clone failed.\n"
        f"Return code: {completed.returncode}\n"
        f"stderr:\n{completed.stderr}"
    )

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

requirements_path = REPO_DIR / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(
        f"requirements.txt was not found at {requirements_path}"
    )

print("\n=== requirements.txt ===")
print(requirements_path.read_text(encoding="utf-8")[:4000])

print("\n=== installing requirements ===")

pip_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(requirements_path),
    ],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
)

print("=== pip stdout ===")
print(pip_result.stdout)

print("=== pip stderr ===")
print(pip_result.stderr)

print("pip return code =", pip_result.returncode)

if pip_result.returncode != 0:
    raise RuntimeError(
        "pip install failed. Check the pip output above."
    )

print("Jiaozi repository:", REPO_DIR)
print("Jiaozi repo URL:", REPO_URL)
print("Jiaozi ref:", REPO_REF)

pipeline_candidates = list(REPO_DIR.rglob("pipeline.py"))

print("pipeline.py candidates:")
for path in pipeline_candidates:
    print(" -", path)

if not pipeline_candidates:
    raise FileNotFoundError(
        "No pipeline.py was found after cloning the repository."
    )


REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...


=== requirements.txt ===
# Runtime dependencies for the Jiaozi CV Auto-DL pipeline.
# Install with the same Python interpreter used to run tests, for example:
#   /usr/bin/python3 -m pip install -r requirements.txt

chromadb
datasets
networkx
numpy
openai>=2.0.0
pandas
pillow
scikit-learn
sentence-transformers
torch
torchvision
transformers

# Test runner for the module4_agent pytest suite.
pytest


=== installing requirements ===
=== pip stdout ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.2 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 83.0.0
    Uninstalling setuptools-83.0.0:
      Successfully uninstalled setuptools-83.0.0

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-5.5')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token


## Download the Plant Pathology data

These cells configure the Kaggle CLI, download `plant-pathology-2020-fgvc7`, and extract the competition files under `/content/data`. The pipeline cell later converts the one-hot `train.csv` into a single `label` column that Jiaozi's local CSV loader can consume.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

# 先升级基础安装工具
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

# 再安装 Kaggle CLI，不要 -q
kaggle_install = run_pip(["install", "--upgrade", "kaggle"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

# 从 Colab Secrets 读取 Kaggle access token
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# 检查 kaggle 是否可用
subprocess.run(["kaggle", "--version"], check=False)


Running: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
=== pip stdout ===
  Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-83.0.0-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.

return code = 0
Running: /usr/bin/python3 -m pip install --upgrade kaggle
=== pip stdout ===

=== pip stderr ===

return code = 0
Kaggle access token configured.


CompletedProcess(args=['kaggle', '--version'], returncode=0)

In [ ]:
# Configure Kaggle and download Dog Breed Identification dataset
KAGGLE_COMPETITION = "dog-breed-identification"
!kaggle competitions download -c {KAGGLE_COMPETITION} -p /content/data/dog-breed


dog-breed-identification.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
from pathlib import Path
import zipfile

DATA_ROOT = Path("/content/data/dog-breed")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_files = sorted(DATA_ROOT.glob(f"{KAGGLE_COMPETITION}*.zip")) or sorted(DATA_ROOT.glob("*.zip"))
print("Zip files:", zip_files)

if not zip_files:
    raise FileNotFoundError("No .zip file found in /content/data/dog-breed")

zip_path = zip_files[0]
KAGGLE_DATA_DIR = DATA_ROOT / zip_path.stem
KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting:", zip_path)
print("To:", KAGGLE_DATA_DIR)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(KAGGLE_DATA_DIR)

# Extract nested archives if any
for nested_zip in sorted(KAGGLE_DATA_DIR.rglob("*.zip")):
    nested_target = nested_zip.with_suffix("")
    nested_target.mkdir(parents=True, exist_ok=True)
    print("Extracting nested:", nested_zip, "->", nested_target)
    with zipfile.ZipFile(nested_zip, "r") as z:
        z.extractall(nested_target)

print("Done.")
print("KAGGLE_DATA_DIR =", KAGGLE_DATA_DIR)
print("Top-level files:")
for p in sorted(KAGGLE_DATA_DIR.iterdir()):
    print(" -", p.name)


Zip files: [PosixPath('/content/data/dog-breed/dog-breed-identification.zip')]
Extracting: /content/data/dog-breed/dog-breed-identification.zip
To: /content/data/dog-breed/dog-breed-identification
Done.
KAGGLE_DATA_DIR = /content/data/dog-breed/dog-breed-identification
Top-level files:
 - jiaozi_train.csv
 - labels.csv
 - sample_submission.csv
 - test
 - train


## Mount Google Drive for caches and checkpoints

HuggingFace caches and training checkpoints can live on Drive so a recycled Colab runtime does not force every artifact to be downloaded or trained from scratch.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF_HOME = /content/drive/MyDrive/Jiaozi/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated pipeline

This cell runs the integrated Jiaozi flow against the local Kaggle files:

1. Convert Plant Pathology's one-hot labels into a local CSV with `image_id,label`.
2. Run Module 1 from the natural-language `QUERY`.
3. Run Module 2 on sampled real images from the Kaggle folder.
4. Run Module 3 retrieval and optional recommender re-ranking.
5. Run Module 4 code generation with local CSV training metadata.

Set `REAL_TRAINING = True` to generate real-training code and train it in the next cell. With `REAL_TRAINING = False`, Module 4 keeps the generated project in smoke-test mode.


In [ ]:
# Dog Breed Kaggle data -> Jiaozi Module 1-4

import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/Jiaozi")
PIPELINE_PATH = REPO_DIR / "pipeline.py"

if not PIPELINE_PATH.exists():
    raise FileNotFoundError(
        f"No pipeline.py found at {PIPELINE_PATH}. "
        "Please rerun the git clone cell first."
    )

# Always run repository-dependent code from the repository root
os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Current working directory:", Path.cwd())
print("Using pipeline:", PIPELINE_PATH)

# -------------------------------------------------
# 1. Dog Breed task prompt
# -------------------------------------------------
QUERY = (
    "Classify dog images into 120 fine-grained dog breeds using a small image dataset. "
    "The task requires distinguishing visually similar breeds. "
    "The official evaluation metric is multiclass log loss, "
    "and prediction probabilities must be well calibrated."
)

REAL_TRAINING = True
EPOCHS = None
OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")

# -------------------------------------------------
# 2. Find Kaggle Dog Breed data
# -------------------------------------------------
DATASET_DIR = Path(KAGGLE_DATA_DIR)

raw_train_csvs = sorted(DATASET_DIR.rglob("labels.csv"))

if not raw_train_csvs:
    raw_train_csvs = sorted(DATASET_DIR.rglob("train.csv"))

if not raw_train_csvs:
    raise FileNotFoundError(
        f"No labels.csv or train.csv found under {DATASET_DIR}"
    )

raw_train_csv = raw_train_csvs[0]
raw_frame = pd.read_csv(raw_train_csv)

print("Raw train csv:", raw_train_csv)
print("Raw columns:", raw_frame.columns.tolist())
print("Raw shape:", raw_frame.shape)

# -------------------------------------------------
# 3. Convert to Jiaozi-friendly CSV
# -------------------------------------------------
if "id" not in raw_frame.columns:
    raise ValueError(
        f"Expected an 'id' column, found: {raw_frame.columns.tolist()}"
    )

if "breed" in raw_frame.columns:
    label_column = "breed"
elif "label" in raw_frame.columns:
    label_column = "label"
else:
    raise ValueError(
        f"Could not identify label column in {raw_frame.columns.tolist()}"
    )

processed_frame = raw_frame[["id", label_column]].copy()
processed_frame = processed_frame.rename(
    columns={
        "id": "image",
        label_column: "label",
    }
)

processed_frame["image"] = (
    processed_frame["image"].astype(str) + ".jpg"
)

processed_train_csv = raw_train_csv.with_name("jiaozi_train.csv")
processed_frame.to_csv(processed_train_csv, index=False)

print("\nProcessed train csv:", processed_train_csv)
print(processed_frame.head())
print("Number of classes:", processed_frame["label"].nunique())

# -------------------------------------------------
# 4. Locate training image directory
# -------------------------------------------------
candidate_image_dirs = [
    raw_train_csv.parent / "train",
    DATASET_DIR / "train",
]

image_dir = None

for candidate in candidate_image_dirs:
    if candidate.exists() and candidate.is_dir():
        image_dir = candidate
        break

if image_dir is None:
    matches = [
        path
        for path in DATASET_DIR.rglob("train")
        if path.is_dir()
    ]

    if matches:
        image_dir = matches[0]

if image_dir is None:
    raise FileNotFoundError(
        f"Could not find the Dog Breed train image directory under {DATASET_DIR}"
    )

print("Image directory:", image_dir)

sample_image = processed_frame.iloc[0]["image"]
sample_path = image_dir / sample_image

if not sample_path.exists():
    raise FileNotFoundError(
        f"Sample image not found: {sample_path}"
    )

print("Sample image exists:", sample_path)

# -------------------------------------------------
# 5. Local benchmark information
# -------------------------------------------------
LOCAL_DATA_INFO = {
    "benchmark": "dog_breed_identification",
    "competition": "dog-breed-identification",
    "train_csv": str(processed_train_csv),
    "image_dir": str(image_dir),
    "image_column": "image",
    "label_column": "label",
    "target_column": "label",
    "image_path_template": "{image}",
    "image_extension": "",
    "num_classes": int(processed_frame["label"].nunique()),
    "metric": "log_loss",
}

print("\nPrepared local Kaggle Dog Breed data:")
print(
    json.dumps(
        LOCAL_DATA_INFO,
        indent=2,
        ensure_ascii=False,
    )
)

# Remove only the generated Module 4 project
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

# -------------------------------------------------
# 6. Prepare a stable ChromaDB directory
# -------------------------------------------------
CHROMA_DIR = Path("/content/jiaozi_chroma_db")
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("ChromaDB directory:", CHROMA_DIR)

# -------------------------------------------------
# 7. Import Jiaozi modules
# -------------------------------------------------
from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import (
    derive_recommended_epochs,
    merge_modules,
    run_module4_generation,
)
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import (
    build_all_task_lists,
    build_graph,
    build_vector_index,
    print_results,
    retrieve_top3_hybrid,
)

# -------------------------------------------------
# 8. Module 1
# -------------------------------------------------
print("\n[Notebook] Module 1: Parsing user requirements...")

m1_output = module1_pipeline(QUERY)

if m1_output is None:
    raise RuntimeError(
        "Module 1 failed, so no Module 3 or Module 4 output was produced."
    )

# -------------------------------------------------
# 9. Local image dataset wrapper
# -------------------------------------------------
class LocalImageSplit:
    column_names = ["image", "label"]

    def __init__(self, frame, info):
        from PIL import Image

        self._Image = Image
        self.labels = frame[info["label_column"]].tolist()
        self.image_paths = []

        image_root = Path(info["image_dir"])
        image_column = info["image_column"]
        label_column = info["label_column"]
        template = info.get("image_path_template", "{image}")
        extension = info.get("image_extension", "")

        for _, row in frame.iterrows():
            image_value = str(row[image_column])

            relative = template.format(
                image=image_value,
                label=str(row[label_column]),
                stem=Path(image_value).stem,
            )

            image_path = image_root / relative

            if extension and not image_path.suffix:
                image_path = image_path.with_suffix(extension)

            if not image_path.is_file():
                raise FileNotFoundError(
                    f"Training image referenced by CSV is missing: {image_path}"
                )

            self.image_paths.append(image_path)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, key):
        if key == "label":
            return self.labels

        if key == "image":
            return [
                self._open_image(path)
                for path in self.image_paths
            ]

        index = int(key)

        return {
            "image": self._open_image(self.image_paths[index]),
            "label": self.labels[index],
        }

    def _open_image(self, path):
        with self._Image.open(path) as image:
            return image.convert("RGB")


def build_local_module2_dataset(info):
    frame = pd.read_csv(info["train_csv"])

    required = {
        info["image_column"],
        info["label_column"],
    }

    missing = required - set(frame.columns)

    if missing:
        raise ValueError(
            f"Missing required CSV columns: {sorted(missing)}"
        )

    return {
        "train": LocalImageSplit(frame, info)
    }


# -------------------------------------------------
# 10. Module 2
# -------------------------------------------------
print(
    "\n[Notebook] Module 2: "
    "Analyzing sampled real Kaggle images..."
)

m2_report = ImageStatisticsAnalyzer().analyze(
    build_local_module2_dataset(LOCAL_DATA_INFO)
)

# -------------------------------------------------
# 11. Merge Module 1 and Module 2
# -------------------------------------------------
m3_input = merge_modules(m1_output, m2_report)

m3_input["evaluation_metric"] = LOCAL_DATA_INFO["metric"]
m3_input["num_classes"] = LOCAL_DATA_INFO["num_classes"]

print(
    f"\n[Notebook] Merged: "
    f"task={m3_input['task_type']} "
    f"size={m3_input['data_size']} "
    f"classes={m3_input.get('num_classes')} "
    f"metric={m3_input['evaluation_metric']}"
)

# -------------------------------------------------
# 12. Module 3
# -------------------------------------------------
print(
    "\n[Notebook] Module 3: "
    "Retrieving model configurations..."
)

graph = build_graph()

# Important Colab fix:
# use an explicit absolute writable path for ChromaDB
collection = build_vector_index(
    persist_path=str(CHROMA_DIR)
)

recommendations = retrieve_top3_hybrid(
    m3_input,
    graph,
    collection,
)

print_results(
    m3_input,
    recommendations,
    graph,
)

try:
    recommendations = recommend(
        recommendations,
        m2_report,
        m3_input,
        memory=OutcomeMemory(),
    )

    print("[Notebook] Recommender re-ranked candidates.")

except Exception as exc:
    print(f"[Notebook] Recommender skipped: {exc}")

# -------------------------------------------------
# 13. Build task lists and inject settings
# -------------------------------------------------
task_lists = build_all_task_lists(
    recommendations,
    graph,
    fmt="nl",
)

if not task_lists:
    raise RuntimeError(
        "Module 3 returned no task lists."
    )

for task_list in task_lists:
    model_config = task_list.get("model_config")

    if not isinstance(model_config, dict):
        continue

    model_config.update(
        {
            "num_classes": LOCAL_DATA_INFO["num_classes"],
            "train_csv": LOCAL_DATA_INFO["train_csv"],
            "image_dir": LOCAL_DATA_INFO["image_dir"],
            "image_column": LOCAL_DATA_INFO["image_column"],
            "label_column": LOCAL_DATA_INFO["label_column"],
            "target_column": LOCAL_DATA_INFO["target_column"],
            "image_path_template": LOCAL_DATA_INFO[
                "image_path_template"
            ],
            "image_extension": LOCAL_DATA_INFO[
                "image_extension"
            ],
            "evaluation_metric": LOCAL_DATA_INFO["metric"],
            "metric": LOCAL_DATA_INFO["metric"],
            "metric_name": LOCAL_DATA_INFO["metric"],
            "offline_smoke": not REAL_TRAINING,
            "benchmark_key": LOCAL_DATA_INFO["benchmark"],
            "competition": LOCAL_DATA_INFO["competition"],
        }
    )

    model_config.setdefault(
        "recommended_epochs",
        derive_recommended_epochs(
            m3_input.get("data_size", "medium"),
            model_config.get("finetune_strategy"),
            bool(model_config.get("pretrained_hf_id")),
        ),
    )

print("\nTop model config preview:")
print(
    json.dumps(
        task_lists[0]["model_config"],
        indent=2,
        ensure_ascii=False,
    )[:2500]
)

# -------------------------------------------------
# 14. Module 4 code generation
# -------------------------------------------------
print(
    "\n[Notebook] Module 4: "
    "Generating real-training project..."
)

module4 = run_module4_generation(
    task_lists,
    OUTPUT_DIR,
    skip_smoke=REAL_TRAINING,
    timeout=120,
    llm_provider="openai",
)

summary_path = OUTPUT_DIR / "module4_summary.json"

# Some workflow versions return the summary
# but do not automatically write this file.
if not summary_path.exists():
    summary_path.write_text(
        json.dumps(
            module4["summary"],
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

DATASET = LOCAL_DATA_INFO["train_csv"]

print("\nModule 4 summary:", summary_path)
print("Generated project output dir:", OUTPUT_DIR)
print("DATASET:", DATASET)


Current working directory: /content/Jiaozi
Using pipeline: /content/Jiaozi/pipeline.py
Raw train csv: /content/data/dog-breed/dog-breed-identification/labels.csv
Raw columns: ['id', 'breed']
Raw shape: (10222, 2)

Processed train csv: /content/data/dog-breed/dog-breed-identification/jiaozi_train.csv
                                  image             label
0  000bec180eb18c7604dcecc8fe0dba07.jpg       boston_bull
1  001513dfcb2ffafc82cccf4d8bbaba97.jpg             dingo
2  001cdf01b096e06d78e9e5112d419397.jpg          pekinese
3  00214f311d5d2247d5dfe4fe24b2303d.jpg          bluetick
4  0021f9ceb3235effd7fcde7f7538ed62.jpg  golden_retriever
Number of classes: 120
Image directory: /content/data/dog-breed/dog-breed-identification/train
Sample image exists: /content/data/dog-breed/dog-breed-identification/train/000bec180eb18c7604dcecc8fe0dba07.jpg

Prepared local Kaggle Dog Breed data:
{
  "benchmark": "dog_breed_identification",
  "competition": "dog-breed-identification",
  "train_csv":

In [ ]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


{
  "candidates": [
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 1,
      "score": 0.82,
      "task_type": "classification"
    },
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 2,
      "score": 0.82,
      "task_type": "classification"
    },
    {
      "backbone": "dinov2",
      "finetune_strategy": "head_only",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 3,
      "score": 0.762,
      "task_type": "classification"
    },
    {
      "backbone": "efficientnet",
      "finetune_strategy": "head_only",
      "loss": "cross_entropy_loss",
      "optimizer": "adam",
      "rank": 4,
      "score": 0.588,
      "task_type": "classification"
    }
  ],
  "errors": [],
  "generated_files": [
    "README_generated.md",
    "configs.json",
    "evalua

## Train the recommended model (real data)

This step only runs when `REAL_TRAINING = True`. It executes the generated `run.py`,
which loads the real dataset, trains the model Module 3 recommended, evaluates on the
test split, and saves checkpoints. Select a GPU runtime first for reasonable speed.


In [ ]:
# Train the recommended Dog Breed model with correct multiclass Log Loss

import json
import os
import subprocess
import sys
from pathlib import Path

if not REAL_TRAINING:
    print("REAL_TRAINING is False - skipping the real training step.")
else:
    SAVE_CHECKPOINTS_TO_DRIVE = True

    cfg_path = OUTPUT_DIR / "configs.json"
    configs = json.loads(cfg_path.read_text(encoding="utf-8"))

    if not configs:
        raise ValueError("configs.json is empty.")

    cfg = configs[0]
    mc = cfg.get("model_config", cfg)

    # -------------------------------------------------
    # Fix: train.py recognizes "log_loss", not "logloss"
    # -------------------------------------------------
    cfg["evaluation_metric"] = "log_loss"
    cfg["metric"] = "log_loss"
    cfg["metric_name"] = "log_loss"

    if isinstance(cfg.get("model_config"), dict):
        cfg["model_config"]["evaluation_metric"] = "log_loss"
        cfg["model_config"]["metric"] = "log_loss"
        cfg["model_config"]["metric_name"] = "log_loss"

    epochs = EPOCHS
    if epochs is None:
        epochs = (
            mc.get("recommended_epochs")
            or cfg.get("recommended_epochs")
            or 10
        )

    backbone = mc.get("backbone") or cfg.get("backbone") or "model"
    dataset_used = (
        mc.get("dataset_id")
        or cfg.get("dataset_id")
        or DATASET
    )

    # -------------------------------------------------
    # Use a new checkpoint folder.
    # Do not resume the old accuracy-based checkpoint.
    # -------------------------------------------------
    if SAVE_CHECKPOINTS_TO_DRIVE and Path("/content/drive/MyDrive").exists():
        ckpt_dir = (
            "/content/drive/MyDrive/Jiaozi/checkpoints/"
            "dog_breed_log_loss_fixed"
        )

        os.makedirs(ckpt_dir, exist_ok=True)

        cfg["checkpoint_dir"] = ckpt_dir
        cfg["resume_checkpoint"] = None

        if isinstance(cfg.get("model_config"), dict):
            cfg["model_config"]["checkpoint_dir"] = ckpt_dir
            cfg["model_config"]["resume_checkpoint"] = None

        print("Checkpoints ->", ckpt_dir)
        print("Resume checkpoint -> disabled")
    elif SAVE_CHECKPOINTS_TO_DRIVE:
        raise RuntimeError(
            "Google Drive is not mounted. Run the Drive mount cell first."
        )
    else:
        ckpt_dir = str(OUTPUT_DIR / "checkpoints_log_loss")

        cfg["checkpoint_dir"] = ckpt_dir
        cfg["resume_checkpoint"] = None

        if isinstance(cfg.get("model_config"), dict):
            cfg["model_config"]["checkpoint_dir"] = ckpt_dir
            cfg["model_config"]["resume_checkpoint"] = None

    # Save the corrected config
    cfg_path.write_text(
        json.dumps(configs, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    # Final check before training
    saved_cfg = json.loads(
        cfg_path.read_text(encoding="utf-8")
    )[0]

    saved_mc = saved_cfg.get("model_config", {})

    print("evaluation_metric:", saved_cfg.get("evaluation_metric"))
    print(
        "model_config evaluation_metric:",
        saved_mc.get("evaluation_metric"),
    )
    print("checkpoint_dir:", saved_cfg.get("checkpoint_dir"))
    print("resume_checkpoint:", saved_cfg.get("resume_checkpoint"))

    assert saved_cfg.get("evaluation_metric") == "log_loss"

    if isinstance(saved_mc, dict) and saved_mc:
        assert saved_mc.get("evaluation_metric") == "log_loss"

    print(
        f"Training {backbone} for {epochs} epochs "
        f"on {dataset_used} ..."
    )

    train_command = [
        sys.executable,
        "-u",
        "run.py",
        "--epochs",
        str(epochs),
    ]

    print(
        "Command:",
        " ".join(train_command),
        "(cwd:",
        OUTPUT_DIR,
        ")",
    )

    env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    train_command,
    cwd=str(OUTPUT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

if process.stdout is None:
    raise RuntimeError("Could not capture training output.")

for line in iter(process.stdout.readline, ""):
    print(line, end="", flush=True)

process.stdout.close()

return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(
        return_code,
        train_command,
    )

    print("\nReal training finished.")
    print("Checkpoints under:", ckpt_dir)


Checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/dog_breed_log_loss_fixed
Resume checkpoint -> disabled
evaluation_metric: log_loss
model_config evaluation_metric: log_loss
checkpoint_dir: /content/drive/MyDrive/Jiaozi/checkpoints/dog_breed_log_loss_fixed
resume_checkpoint: None
Training dinov3 for 40 epochs on /content/data/dog-breed/dog-breed-identification/jiaozi_train.csv ...
Command: /usr/bin/python3 -u run.py --epochs 40 (cwd: /content/jiaozi_generated_pipeline )

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 1258.15it/s]
[train] requested_backbone=dinov3 hf_id=facebook/dinov3-vitb16-pretrain-lvd1689m source=huggingface actual_model=facebook/dinov3-vitb16-pretrain-lvd1689m backbone_class=_HFBackbone feature_pooling=pooler_output_or_cls_token
[train] params total=85752696 trainable=130680 backbone_trainable=38400 head_trainable=92280 other_trainable=0
[train] finetune strategy=partial unfreeze_last_n_blocks=2 frozen_backbone_param_tensors=211 partial_unfrozen_par

KeyboardInterrupt: 

In [ ]:
!nvidia-smi


Sat Jul 11 18:36:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             33W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Show the result

Reads the best checkpoint and prints its validation score **instantly** (the metric was
already computed during training — no re-evaluation). Set `FULL_EVAL = True` for a full
re-evaluation (macro-F1, params, sample count), which costs one extra pass over the eval set.


In [ ]:
import json, torch
from pathlib import Path

# Locate best_model.pt (prefer the Drive checkpoint dir set by the training cell).
_candidates = []
try:
    _candidates.append(Path(ckpt_dir) / 'best_model.pt')
except NameError:
    pass
_candidates.append(Path(OUTPUT_DIR) / 'checkpoints' / 'best_model.pt')
best_path = next((p for p in _candidates if p.exists()), None)
if best_path is None:
    raise FileNotFoundError(f'best_model.pt not found in: {[str(p) for p in _candidates]}')

ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
print('=== RESULT (best checkpoint) ===')
print('checkpoint :', best_path)
print('best_epoch :', ckpt.get('best_epoch'))
print('best_metric:', ckpt.get('best_metric'), '(validation metric at best epoch — no re-eval)')
print('validation :', json.dumps(ckpt.get('validation'), ensure_ascii=False))

# Full re-evaluation (macro_f1, params, num_samples) — one extra pass over the eval set.
FULL_EVAL = False
if FULL_EVAL:
    import os, sys
    os.chdir(OUTPUT_DIR); sys.path.insert(0, str(OUTPUT_DIR))
    from model import build_model
    from evaluate import evaluate
    _cfg = json.loads((Path(OUTPUT_DIR) / 'configs.json').read_text(encoding='utf-8'))[0]
    _cfg.update(_cfg.pop('model_config', {}) or {})
    _model = build_model(_cfg); _model.load_state_dict(ckpt['model_state_dict'])
    print('\n=== FULL EVALUATE ===')
    print(json.dumps(evaluate(_model, _cfg), indent=2, ensure_ascii=False))


=== RESULT (best checkpoint) ===
checkpoint : /content/drive/MyDrive/Jiaozi/checkpoints/dog_breed_log_loss_fixed/best_model.pt
best_epoch : 7
best_metric: 0.437158644093814 (validation metric at best epoch — no re-eval)
validation : {"metric_name": "log_loss", "metric_value": 0.437158644093814, "accuracy": 0.8640156984329224, "epoch": 7}


In [ ]:
# [替换 Cell 17] Generate Kaggle Dog Breed Submission.csv

import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
# FIX 1: Point to Dog Breed Competition Dir
DATA_DIR = Path("/content/data/dog-breed-identification")

sample_path = DATA_DIR / "sample_submission.csv"
submission_path = OUTPUT_DIR / "submission.csv"

assert OUTPUT_DIR.exists(), OUTPUT_DIR
assert sample_path.exists(), sample_path

os.chdir(OUTPUT_DIR)
if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))

from model import build_model
from train import _build_image_transform

# -----------------------------
# 1. Load config
# -----------------------------
configs = json.loads((OUTPUT_DIR / "configs.json").read_text(encoding="utf-8"))
cfg = configs[0]

if "model_config" in cfg:
    flat_cfg = {**cfg, **cfg["model_config"]}
else:
    flat_cfg = cfg

# Force Dog Breed classification settings
flat_cfg["num_classes"] = 120
flat_cfg["metric_name"] = "log_loss"
flat_cfg["evaluation_metric"] = "log_loss"

print("Config preview:")
for k in ["backbone", "pretrained_hf_id", "model_name", "num_classes", "metric_name"]:
    if k in flat_cfg:
        print(k, ":", flat_cfg[k])

# -----------------------------
# 2. Find best checkpoint
# -----------------------------
ckpt_candidates = []

try:
    ckpt_candidates.append(Path(ckpt_dir) / "best_model.pt")
except NameError:
    pass

if flat_cfg.get("checkpoint_dir"):
    ckpt_candidates.append(Path(flat_cfg["checkpoint_dir"]) / "best_model.pt")

ckpt_candidates.append(OUTPUT_DIR / "checkpoints" / "best_model.pt")
ckpt_candidates.append(OUTPUT_DIR / "best_model.pt")

best_path = next((p for p in ckpt_candidates if p.exists()), None)

if best_path is None:
    raise FileNotFoundError(
        "best_model.pt not found. Checked:\n" + "\n".join(str(p) for p in ckpt_candidates)
    )

print("Using checkpoint:", best_path)

# -----------------------------
# 3. Load model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_model(flat_cfg).to(device)

ckpt = torch.load(best_path, map_location=device, weights_only=False)

if "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
elif "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

model.load_state_dict(state_dict)
model.eval()

# -----------------------------
# 4. Build test transform
# -----------------------------
transform = _build_image_transform(flat_cfg, "test")

# -----------------------------
# 5. Read Dog Breed test ids and sample submission
# -----------------------------
sample = pd.read_csv(sample_path)

# Dynamic mapping of ID column (Kaggle Dog Breed uses "id" instead of "id_code")
id_col = sample.columns[0]
test_ids = sample[id_col].astype(str).tolist()
class_cols = list(sample.columns[1:])  # 120 breeds

# Dynamically locate the test images folder in case of nested ZIP archives
sample_test_img = test_ids[0]
img_matches = list(DATA_DIR.rglob(f"{sample_test_img}.jpg"))
if not img_matches:
    raise FileNotFoundError(f"Could not locate test image {sample_test_img}.jpg under {DATA_DIR}")
image_dir = img_matches[0].parent
print("Dynamic resolved test image folder:", image_dir)

# -----------------------------
# 6. Predict
# -----------------------------
rows = []
batch = []
batch_ids = []

def flush():
    global batch, batch_ids, rows

    if not batch:
        return

    x = torch.stack(batch).to(device)

    with torch.no_grad():
        out = model(x)

        if isinstance(out, dict):
            logits = out.get("logits", next(iter(out.values())))
        elif isinstance(out, (tuple, list)):
            logits = out[0]
        else:
            logits = out

        # For Multi-class Log Loss submission, Kaggle expects class probabilities
        probs = torch.softmax(logits, dim=1).cpu().numpy()

    for image_id, p in zip(batch_ids, probs):
        rows.append([image_id, *p.tolist()])

    batch = []
    batch_ids = []

for image_id in test_ids:
    img_path = image_dir / f"{image_id}.jpg"

    if not img_path.exists():
        raise FileNotFoundError(img_path)

    img = Image.open(img_path).convert("RGB")
    batch.append(transform(img))
    batch_ids.append(image_id)

    if len(batch) >= 64:
        flush()

flush()

# -----------------------------
# 7. Save submission
# -----------------------------
sub = pd.DataFrame(rows, columns=[id_col, *class_cols])

# Keep the same order as sample_submission
sub = sample[[id_col]].merge(sub, on=id_col, how="left")

# Sanity checks
assert len(sub) == len(sample), f"submission rows {len(sub)} != sample rows {len(sample)}"
assert sub[class_cols].notna().all().all()
assert ((sub[class_cols] >= 0) & (sub[class_cols] <= 1)).all().all()
assert abs(sub[class_cols].sum(axis=1).sub(1)).max() < 1e-4

sub.to_csv(submission_path, index=False)

print("Wrote submission:", submission_path)
print("Shape:", sub.shape)
display(sub.head())


In [ ]:
# Submit to Kaggle under dog breed identification
!kaggle competitions submit \
  -c dog-breed-identification \
  -f /content/jiaozi_generated_pipeline/submission.csv \
  -m "Jiaozi Dog Breed late submission"


In [ ]:
!kaggle competitions submissions -c dog-breed-identification
